In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, confusion_matrix
import time

In [2]:
class VGG13(nn.Module):
    """
    VGG-13 (Version B) Architecture adapted for 28x28 images
    
    Original VGG-13 Configuration B:
    - Input: 224x224x3
    - Conv layers: [64, 64] -> [128, 128] -> [256, 256] -> [512, 512] -> [512, 512]
    - MaxPool after each group
    - FC: 4096 -> 4096 -> 1000
    
    Adapted for 28x28x1:
    - Removed some max pooling layers to prevent dimension collapse
    - Adjusted FC layer sizes
    - Changed output to num_classes (36)
    """
    
    def __init__(self, num_classes=36, dropout=0.5):
        super(VGG13, self).__init__()
        
        # Convolutional layers (feature extraction)
        self.features = nn.Sequential(
            # Block 1: 64 filters (28x28 -> 14x14)
            nn.Conv2d(1, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 28x28 -> 14x14
            
            # Block 2: 128 filters (14x14 -> 7x7)
            nn.Conv2d(64, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(128, 128, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 14x14 -> 7x7
            
            # Block 3: 256 filters (keep 7x7, no pooling)
            nn.Conv2d(128, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(256, 256, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Skip MaxPool to maintain spatial dimensions
            
            # Block 4: 512 filters (7x7 -> 3x3)
            nn.Conv2d(256, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.MaxPool2d(kernel_size=2, stride=2),  # 7x7 -> 3x3
            
            # Block 5: 512 filters (keep 3x3)
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            nn.Conv2d(512, 512, kernel_size=3, padding=1),
            nn.ReLU(inplace=True),
            # Skip MaxPool to avoid too small dimensions
        )
        
        # Adaptive pooling to handle varying input sizes
        self.avgpool = nn.AdaptiveAvgPool2d((3, 3))
        
        # Classifier (fully connected layers)
        self.classifier = nn.Sequential(
            nn.Linear(512 * 3 * 3, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(4096, 4096),
            nn.ReLU(inplace=True),
            nn.Dropout(dropout),
            nn.Linear(4096, num_classes)
        )
        
        # Initialize weights
        self._initialize_weights()
    
    def forward(self, x):
        x = self.features(x)
        x = self.avgpool(x)
        x = torch.flatten(x, 1)
        x = self.classifier(x)
        return x
    
    def _initialize_weights(self):
        """Initialize weights as in the original VGG paper"""
        for m in self.modules():
            if isinstance(m, nn.Conv2d):
                nn.init.kaiming_normal_(m.weight, mode='fan_out', nonlinearity='relu')
                if m.bias is not None:
                    nn.init.constant_(m.bias, 0)
            elif isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, 0, 0.01)
                nn.init.constant_(m.bias, 0)

In [3]:
device = torch.device("mps" if torch.backends.mps.is_available()
                      else "cuda" if torch.cuda.is_available()
                      else "cpu")
print(f"Using device: {device}")

# Data transforms (same as Part 3)
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((28, 28)),
    transforms.ToTensor(),
    transforms.Normalize((0.5,), (0.5,))
])

# Load dataset
data_dir = "datasets/cnn_dataset"  # Change to your path
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

print(f"Total samples: {len(full_dataset)}")
print(f"Number of classes: {len(full_dataset.classes)}")

# Split dataset (70:15:15)
train_size = int(0.7 * len(full_dataset))
val_size = int(0.15 * len(full_dataset))
test_size = len(full_dataset) - train_size - val_size

train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(
    full_dataset, [train_size, val_size, test_size],
    generator=torch.Generator().manual_seed(42)
)

print(f"\nDataset splits:")
print(f"  Training: {len(train_dataset)}")
print(f"  Validation: {len(val_dataset)}")
print(f"  Testing: {len(test_dataset)}")

# Create dataloaders
batch_size = 64
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True, num_workers=0)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False, num_workers=0)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False, num_workers=0)

Using device: mps
Total samples: 100800
Number of classes: 36

Dataset splits:
  Training: 70560
  Validation: 15120
  Testing: 15120


In [4]:
# Create VGG-13 model
vgg13_model = VGG13(num_classes=len(full_dataset.classes), dropout=0.5).to(device)

# Print model architecture
print("\nVGG-13 Architecture:")
print(vgg13_model)

# Count parameters
total_params = sum(p.numel() for p in vgg13_model.parameters())
trainable_params = sum(p.numel() for p in vgg13_model.parameters() if p.requires_grad)
print(f"\nTotal parameters: {total_params:,}")
print(f"Trainable parameters: {trainable_params:,}")


VGG-13 Architecture:
VGG13(
  (features): Sequential(
    (0): Conv2d(1, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (1): ReLU(inplace=True)
    (2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (3): ReLU(inplace=True)
    (4): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (5): Conv2d(64, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (6): ReLU(inplace=True)
    (7): Conv2d(128, 128, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (8): ReLU(inplace=True)
    (9): MaxPool2d(kernel_size=2, stride=2, padding=0, dilation=1, ceil_mode=False)
    (10): Conv2d(128, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (11): ReLU(inplace=True)
    (12): Conv2d(256, 256, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (13): ReLU(inplace=True)
    (14): Conv2d(256, 512, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1))
    (15): ReLU(inplace=True)
    (16): Conv2d(512, 512, kernel_size=(3,

In [5]:
def train_epoch_vgg(model, dataloader, loss_fn, optimizer, device):
    """Train for one epoch"""
    model.train()
    total_loss, correct, total = 0, 0, 0
    
    for batch_idx, (images, labels) in enumerate(dataloader):
        images, labels = images.to(device), labels.to(device)
        
        # Forward pass
        outputs = model(images)
        loss = loss_fn(outputs, labels)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        # Track metrics
        total_loss += loss.item() * images.size(0)
        _, predicted = torch.max(outputs, 1)
        correct += (predicted == labels).sum().item()
        total += labels.size(0)
        
        # Print progress every 100 batches
        if (batch_idx + 1) % 100 == 0:
            print(f"  Batch [{batch_idx+1}/{len(dataloader)}] - "
                  f"Loss: {loss.item():.4f}")
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

def validate_epoch_vgg(model, dataloader, loss_fn, device):
    """Validate for one epoch"""
    model.eval()
    total_loss, correct, total = 0, 0, 0
    
    with torch.no_grad():
        for images, labels in dataloader:
            images, labels = images.to(device), labels.to(device)
            
            outputs = model(images)
            loss = loss_fn(outputs, labels)
            
            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
    
    avg_loss = total_loss / total
    accuracy = correct / total
    return avg_loss, accuracy

In [6]:
def train_vgg13(model, train_loader, val_loader, loss_fn, optimizer, 
                scheduler, epochs=50, device='cpu'):
    """
    Training loop with learning rate scheduler (as in VGG paper)
    """
    history = {
        'train_loss': [], 'val_loss': [],
        'train_acc': [], 'val_acc': [],
        'epoch_time': [], 'lr': []
    }
    
    best_val_acc = 0.0
    best_model_state = None
    
    print("\nStarting training...")
    print(f"Initial learning rate: {optimizer.param_groups[0]['lr']}")
    
    for epoch in range(1, epochs + 1):
        start_time = time.time()
        
        print(f"\nEpoch {epoch}/{epochs}")
        print("-" * 40)
        
        # Train
        train_loss, train_acc = train_epoch_vgg(model, train_loader, loss_fn, optimizer, device)
        
        # Validate
        val_loss, val_acc = validate_epoch_vgg(model, val_loader, loss_fn, device)
        
        # Step scheduler (reduce LR on plateau)
        scheduler.step(val_loss)
        current_lr = optimizer.param_groups[0]['lr']
        
        epoch_time = time.time() - start_time
        
        # Save history
        history['train_loss'].append(train_loss)
        history['val_loss'].append(val_loss)
        history['train_acc'].append(train_acc)
        history['val_acc'].append(val_acc)
        history['epoch_time'].append(epoch_time)
        history['lr'].append(current_lr)
        
        # Save best model
        if val_acc > best_val_acc:
            best_val_acc = val_acc
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            print(f"  ✓ New best model! Val Acc: {val_acc:.4f}")
        
        # Print epoch summary
        print(f"  Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f}")
        print(f"  Val Loss: {val_loss:.4f} | Val Acc: {val_acc:.4f}")
        print(f"  Learning Rate: {current_lr:.6f}")
        print(f"  Time: {epoch_time:.2f}s")
    
    # Load best model
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})
        print(f"\n✓ Loaded best model with validation accuracy: {best_val_acc:.4f}")
    
    print(f"\nTotal training time: {sum(history['epoch_time']):.2f}s")
    
    return history

In [8]:
# Loss function
loss_fn = nn.CrossEntropyLoss()

# Optimizer (as in VGG paper: SGD with momentum)
# You can also use Adam for potentially faster convergence
optimizer = torch.optim.SGD(
    vgg13_model.parameters(),
    lr=0.01,          # Initial learning rate
    momentum=0.9,      # Momentum (as in VGG paper)
    weight_decay=5e-4  # L2 regularization
)

# Learning rate scheduler (as required in assignment)
# ReduceLROnPlateau: reduces LR when validation loss plateaus
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='min',        # Minimize validation loss
    factor=0.1,        # Reduce LR by factor of 10
    patience=5,        # Wait 5 epochs before reducing
)

# Alternative: Use Adam optimizer (often converges faster)
# optimizer = torch.optim.Adam(vgg13_model.parameters(), lr=0.001)

# Train the model
vgg_history = train_vgg13(
    vgg13_model, train_loader, val_loader, loss_fn, optimizer, scheduler,
    epochs=50, device=device
)


Starting training...
Initial learning rate: 0.01

Epoch 1/50
----------------------------------------
  Batch [100/1103] - Loss: 3.5791
  Batch [200/1103] - Loss: 2.8955
  Batch [300/1103] - Loss: 1.7066
  Batch [400/1103] - Loss: 0.8415
  Batch [500/1103] - Loss: 0.6211
  Batch [600/1103] - Loss: 0.4118
  Batch [700/1103] - Loss: 0.7255
  Batch [800/1103] - Loss: 0.4104
  Batch [900/1103] - Loss: 0.4502
  Batch [1000/1103] - Loss: 0.4595
  Batch [1100/1103] - Loss: 0.5908
  ✓ New best model! Val Acc: 0.8690
  Train Loss: 1.3009 | Train Acc: 0.6125
  Val Loss: 0.3575 | Val Acc: 0.8690
  Learning Rate: 0.010000
  Time: 217.87s

Epoch 2/50
----------------------------------------
  Batch [100/1103] - Loss: 0.3111
  Batch [200/1103] - Loss: 0.2258
  Batch [300/1103] - Loss: 0.2530
  Batch [400/1103] - Loss: 0.2814
  Batch [500/1103] - Loss: 0.3020
  Batch [600/1103] - Loss: 0.3671
  Batch [700/1103] - Loss: 0.2572
  Batch [800/1103] - Loss: 0.5449
  Batch [900/1103] - Loss: 0.3555
  Batc

KeyboardInterrupt: 